# 🌟 Tutorial 04: MAML - Model-Agnostic Meta-Learning

## El Algoritmo Estrella del Meta-Learning

En este tutorial aprenderás:

- 🎯 Qué es MAML y por qué es revolucionario
- 🔄 Inner loop vs Outer loop optimization
- 📐 Gradientes de segundo orden
- 💻 Implementación completa de MAML

---

## 📖 Parte 1: Teoría

### ¿Qué es MAML?

**MAML** (Finn et al., 2017) es un algoritmo que aprende una **inicialización óptima** de los parámetros del modelo. Esta inicialización está optimizada específicamente para permitir **adaptación rápida** a nuevas tareas con pocos pasos de gradient descent.

### La Gran Idea:

> "No busques parámetros que funcionen bien en promedio, busca parámetros desde los cuales sea fácil aprender."

### Dos Niveles de Optimización:

#### 1️⃣ **Inner Loop** (Task-specific adaptation):
- Para cada tarea $\mathcal{T}_i$, adaptamos los parámetros $\theta$ con K pasos de gradient descent:
$$\theta'_i = \theta - \alpha \nabla_{\theta} \mathcal{L}_{\mathcal{T}_i}^{support}(\theta)$$

#### 2️⃣ **Outer Loop** (Meta-optimization):
- Actualizamos la inicialización $\theta$ para minimizar el loss en query sets después de adaptación:
$$\theta \leftarrow \theta - \beta \nabla_{\theta} \sum_{\mathcal{T}_i} \mathcal{L}_{\mathcal{T}_i}^{query}(\theta'_i)$$

### Visualización:

```
       θ (Meta-parameters)
       |
       |-- Task 1: θ → θ'₁ (inner loop) → evaluate on query
       |-- Task 2: θ → θ'₂ (inner loop) → evaluate on query
       |-- Task 3: θ → θ'₃ (inner loop) → evaluate on query
       |
       Update θ based on all query losses (outer loop)
```

### ¿Por qué funciona?

MAML encuentra un punto en el espacio de parámetros desde el cual:
- Un pequeño paso de gradient descent lleva a buenas soluciones para cualquier tarea
- La "geometría" del loss landscape es favorable para adaptación


## 📑 Table of Contents- [1 - Theoretical Background](#1)    - [1.1 - The Core Idea of MAML](#1-1)    - [1.2 - Mathematical Formulation](#1-2)    - [1.3 - Inner Loop vs Outer Loop](#1-3)    - [1.4 - Second-Order Gradients](#1-4)- [2 - Setup and Imports](#2)- [3 - Exercise 1 - Inner Loop Adaptation](#ex-1)- [4 - Exercise 2 - Outer Loop Update](#ex-2)- [5 - Exercise 3 - Complete MAML](#ex-3)- [6 - Training MAML](#6)- [7 - Evaluation Protocol](#7)- [8 - Experiments](#8)    - [8.1 - Effect of Inner Steps](#8-1)    - [8.2 - Effect of Inner LR](#8-2)    - [8.3 - First-Order vs Second-Order](#8-3)- [9 - Visualization: Optimization Trajectory](#9)- [10 - MAML vs Prototypical Networks](#10)- [11 - Summary](#11)

---

## 🛠️ Parte 2: Setup

<a name='1-3'></a>### 1.3 - Inner Loop vs Outer Loop ExplainedMAML has a **bi-level optimization** structure:**Inner Loop (Task-Specific Adaptation):**```python# For each task T_i:θ'_i = θ - α * ∇_θ L_T_i(θ)  # Adapt to task using support set```- Uses **support set** of task T_i- Learning rate: **α** (inner learning rate, typically 0.01-0.1)- Takes **K steps** (typically 1-5 steps)- Creates task-specific parameters θ'_i**Outer Loop (Meta-Learning):**```python# Update meta-parameters using ALL tasks:θ = θ - β * ∇_θ Σ_i L_T_i(θ'_i)  # Meta-update using query sets```- Uses **query sets** of all tasks- Learning rate: **β** (outer learning rate, typically 0.001)- Updates meta-initialization θ- Goal: Find θ that adapts quickly to any task### Visual Representation:```Meta-Parameters (θ)        |        | Sample Task T_i        v    [Support Set]        |        | K inner steps with LR=α        v  Adapted Parameters (θ'_i)        |        | Evaluate on        v    [Query Set]        |        | Compute Loss        v      Loss_i        |        | Backprop through        | adaptation steps        v    Meta-Gradient        |        | Update with LR=β        v  Updated θ (better initialization)```**Key Insight**: We're not just learning parameters, we're learning an **initialization** that can quickly adapt to new tasks!

<a name='1-4'></a>### 1.4 - Second-Order Gradients (The Tricky Part)MAML requires computing **gradients of gradients**!**Why Second-Order?**In the outer loop, we need:$$\frac{\partial L_{T_i}(\theta'_i)}{\partial \theta}$$But θ'_i depends on θ:$$\theta'_i = \theta - \alpha \nabla_{\theta} L_{T_i}(\theta)$$So we need the **chain rule**:$$\frac{\partial L_{T_i}(\theta'_i)}{\partial \theta} = \frac{\partial L_{T_i}(\theta'_i)}{\partial \theta'_i} \cdot \frac{\partial \theta'_i}{\partial \theta}$$The second term is the **gradient of a gradient** → second-order!**Computational Cost:**- Second-order: O(n²) memory and compute- First-order approximation: O(n) - ignores second term**PyTorch Implementation:**```python# Second-order (full MAML)grads = torch.autograd.grad(loss, params, create_graph=True)  # create_graph=True!# First-order (FOMAML - faster approximation)grads = torch.autograd.grad(loss, params, create_graph=False)  # Cheaper!```**Trade-off:**- Second-order: Better performance, slower training- First-order: Slightly worse performance, 2-3x faster

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import sys
sys.path.append('..')

# Importar higher para MAML eficiente
try:
    import higher
    HIGHER_AVAILABLE = True
except ImportError:
    HIGHER_AVAILABLE = False
    print("⚠️  La librería 'higher' no está instalada. Usaremos implementación manual.")

from utils.test_utils import print_success, print_hint, HintSystem, run_test
from utils.data_utils import create_sine_task, set_seed
from utils.visualization import plot_few_shot_results, plot_learning_curves

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Dispositivo: {device}")
print("✅ Setup completo!")

---

## 💻 Parte 3: Modelo para Regresión

In [ ]:
class SineModel(nn.Module):
    """Modelo simple para regresión de funciones seno."""
    
    def __init__(self, hidden_size=40):
        super(SineModel, self).__init__()
        self.fc1 = nn.Linear(1, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print("✅ Modelo definido!")

---

## 💻 Parte 4: Ejercicio 1 - Inner Loop de MAML

El inner loop adapta el modelo a una tarea específica.

**Tu tarea**: Completa la función del inner loop.

In [ ]:
def inner_loop(model, task, inner_lr=0.01, inner_steps=5):
    """
    Adapta el modelo a una tarea usando K pasos de gradient descent.
    
    Args:
        model: Modelo a adaptar
        task: Diccionario con x_support, y_support
        inner_lr: Learning rate para adaptación
        inner_steps: Número de pasos de gradient descent
    
    Returns:
        adapted_model: Modelo adaptado (copia)
    """
    # TODO: Implementa el inner loop
    # 1. Crea una copia del modelo (usa deepcopy)
    # 2. Crea un optimizador SGD para la copia
    # 3. Entrena la copia por inner_steps pasos en el support set
    # 4. Retorna el modelo adaptado
    
    pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_inner = HintSystem([
    "Usa deepcopy(model) para crear una copia independiente del modelo.",
    "El inner loop es entrenamiento estándar: loop de inner_steps con zero_grad, forward, loss, backward, step.",
    "Usa MSELoss para regresión y entrena solo en x_support, y_support.",
    "Código: adapted = deepcopy(model); opt = SGD(adapted.parameters(), lr=inner_lr); for _ in range(inner_steps): [train]; return adapted"
])

In [ ]:
# Para ver pistas
hints_inner.show_hint()

In [ ]:
# ✅ TEST 1: Verificar inner loop

def test_inner_loop():
    model = SineModel()
    task = create_sine_task(k_shot=10, q_query=10)
    
    # Evaluar antes de adaptar
    with torch.no_grad():
        pred_before = model(task['x_query'])
        loss_before = F.mse_loss(pred_before, task['y_query']).item()
    
    # Adaptar
    adapted_model = inner_loop(model, task, inner_lr=0.01, inner_steps=5)
    
    # Evaluar después
    with torch.no_grad():
        pred_after = adapted_model(task['x_query'])
        loss_after = F.mse_loss(pred_after, task['y_query']).item()
    
    # Verificar que el modelo se adaptó
    assert adapted_model is not None, "inner_loop debe retornar un modelo"
    assert loss_after < loss_before, f"Loss debe disminuir después de adaptación (antes: {loss_before:.4f}, después: {loss_after:.4f})"
    
    print_success(f"✅ Inner loop funciona! Loss antes: {loss_before:.4f}, después: {loss_after:.4f}")

run_test(test_inner_loop, "Test de Inner Loop")

---

## 💻 Parte 5: Ejercicio 2 - MAML Completo (Versión Simplificada)

Ahora implementaremos MAML completo. Esta es una versión simplificada que no usa gradientes de segundo orden.

**Tu tarea**: Completa el meta-training loop.

In [ ]:
def train_maml_simple(model, n_iterations=1000, meta_batch_size=4, 
                      inner_lr=0.01, outer_lr=0.001, inner_steps=5):
    """
    Entrena modelo con MAML (versión simplificada, sin gradientes de 2do orden).
    
    Args:
        model: Modelo a meta-entrenar
        n_iterations: Número de meta-iteraciones
        meta_batch_size: Número de tareas por iteración
        inner_lr: LR para inner loop
        outer_lr: LR para outer loop
        inner_steps: Pasos de adaptación por tarea
    
    Returns:
        meta_losses: Lista de losses por iteración
    """
    meta_optimizer = optim.Adam(model.parameters(), lr=outer_lr)
    criterion = nn.MSELoss()
    
    meta_losses = []
    
    print(f"🚀 Meta-entrenando por {n_iterations} iteraciones...\n")
    
    for iteration in range(n_iterations):
        meta_loss = 0.0
        
        # TODO: Implementa el meta-training loop
        # Para cada tarea en el meta-batch:
        #   1. Sample una tarea
        #   2. Adapta el modelo (inner loop)
        #   3. Evalúa el modelo adaptado en el query set
        #   4. Acumula el loss
        # 
        # Después de todas las tareas:
        #   5. Calcula el promedio del meta-loss
        #   6. Actualiza el modelo original (outer loop)
        
        pass  # TODO: Reemplaza con tu código
        
        if (iteration + 1) % 100 == 0:
            avg_loss = np.mean(meta_losses[-100:])
            print(f"Iteración {iteration+1}/{n_iterations} - Meta-Loss: {avg_loss:.4f}")
    
    return meta_losses


# Sistema de pistas
hints_maml = HintSystem([
    "El loop externo itera sobre iteraciones, el interno sobre meta_batch_size tareas.",
    "Para cada tarea: task = create_sine_task(); adapted = inner_loop(model, task); query_loss = criterion(adapted(x_query), y_query).",
    "Acumula meta_loss y al final: meta_loss /= meta_batch_size; meta_optimizer.zero_grad(); meta_loss.backward(); meta_optimizer.step().",
    "Importante: Necesitas mantener el grafo computacional entre inner y outer loop para que backward funcione."
])

In [ ]:
# Para ver pistas
hints_maml.show_hint()

---

## 📊 Parte 6: MAML con la Librería 'higher' (Implementación Correcta)

La librería `higher` nos permite implementar MAML correctamente con gradientes de segundo orden.

In [ ]:
if HIGHER_AVAILABLE:
    def train_maml_higher(model, n_iterations=1000, meta_batch_size=4,
                          inner_lr=0.01, outer_lr=0.001, inner_steps=5):
        """
        MAML con higher (gradientes de segundo orden correctos).
        """
        meta_optimizer = optim.Adam(model.parameters(), lr=outer_lr)
        criterion = nn.MSELoss()
        
        meta_losses = []
        
        print(f"🚀 Meta-entrenando con higher por {n_iterations} iteraciones...\n")
        
        for iteration in range(n_iterations):
            meta_optimizer.zero_grad()
            meta_loss = 0.0
            
            for _ in range(meta_batch_size):
                # Sample task
                task = create_sine_task(k_shot=10, q_query=10)
                x_support, y_support = task['x_support'], task['y_support']
                x_query, y_query = task['x_query'], task['y_query']
                
                # Inner loop con higher
                with higher.innerloop_ctx(model, meta_optimizer, 
                                         copy_initial_weights=False) as (fmodel, diffopt):
                    # Adaptación (inner loop)
                    for _ in range(inner_steps):
                        support_pred = fmodel(x_support)
                        support_loss = criterion(support_pred, y_support)
                        diffopt.step(support_loss)
                    
                    # Evaluación en query (outer loop)
                    query_pred = fmodel(x_query)
                    query_loss = criterion(query_pred, y_query)
                    meta_loss += query_loss
            
            # Meta-update
            meta_loss = meta_loss / meta_batch_size
            meta_loss.backward()
            meta_optimizer.step()
            
            meta_losses.append(meta_loss.item())
            
            if (iteration + 1) % 100 == 0:
                avg_loss = np.mean(meta_losses[-100:])
                print(f"Iteración {iteration+1}/{n_iterations} - Meta-Loss: {avg_loss:.4f}")
        
        return meta_losses
    
    # Entrenar con MAML
    maml_model = SineModel().to(device)
    maml_losses = train_maml_higher(maml_model, n_iterations=500, meta_batch_size=4)
    
    print("\n✅ Meta-entrenamiento completado!")
else:
    print("⚠️  Instala 'higher' para entrenar con la implementación correcta de MAML.")
    print("   Ejecuta: pip install higher")

---

## 📊 Parte 7: Evaluación y Comparación

Comparemos MAML con entrenamiento tradicional.

In [ ]:
if HIGHER_AVAILABLE:
    # Crear nueva tarea de test
    test_task = create_sine_task(k_shot=10, q_query=50)
    
    # 1. MAML: Adaptación rápida
    maml_adapted = deepcopy(maml_model)
    optimizer = optim.SGD(maml_adapted.parameters(), lr=0.01)
    maml_curve = []
    
    # Evaluación inicial
    with torch.no_grad():
        pred = maml_adapted(test_task['x_query'])
        loss = F.mse_loss(pred, test_task['y_query']).item()
        maml_curve.append(loss)
    
    # Adaptar por 50 pasos
    for step in range(50):
        optimizer.zero_grad()
        pred = maml_adapted(test_task['x_support'])
        loss = F.mse_loss(pred, test_task['y_support'])
        loss.backward()
        optimizer.step()
        
        # Evaluar en query
        with torch.no_grad():
            pred = maml_adapted(test_task['x_query'])
            loss = F.mse_loss(pred, test_task['y_query']).item()
            maml_curve.append(loss)
    
    # 2. ML Tradicional: Desde cero
    scratch_model = SineModel().to(device)
    optimizer = optim.SGD(scratch_model.parameters(), lr=0.01)
    scratch_curve = []
    
    with torch.no_grad():
        pred = scratch_model(test_task['x_query'])
        loss = F.mse_loss(pred, test_task['y_query']).item()
        scratch_curve.append(loss)
    
    for step in range(50):
        optimizer.zero_grad()
        pred = scratch_model(test_task['x_support'])
        loss = F.mse_loss(pred, test_task['y_support'])
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            pred = scratch_model(test_task['x_query'])
            loss = F.mse_loss(pred, test_task['y_query']).item()
            scratch_curve.append(loss)
    
    # Visualizar
    plt.figure(figsize=(12, 5))
    plt.plot(scratch_curve, label='ML Tradicional (Random Init)', color='red', linewidth=2)
    plt.plot(maml_curve, label='MAML (Meta-learned Init)', color='green', linewidth=2)
    plt.xlabel('Pasos de Adaptación', fontsize=12)
    plt.ylabel('Loss en Query Set', fontsize=12)
    plt.title('MAML vs ML Tradicional: Velocidad de Adaptación', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\n📊 Comparación:")
    print(f"  ML Tradicional - Loss inicial: {scratch_curve[0]:.4f}, Final: {scratch_curve[-1]:.4f}")
    print(f"  MAML - Loss inicial: {maml_curve[0]:.4f}, Final: {maml_curve[-1]:.4f}")
    print(f"\n🎯 MAML es {scratch_curve[10] / maml_curve[10]:.1f}x mejor después de 10 pasos!")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **MAML** optimiza la inicialización para adaptación rápida
2. Usa **dos niveles de optimización**: inner loop (adaptación) y outer loop (meta-learning)
3. Requiere **gradientes de segundo orden** para funcionar correctamente
4. La librería **higher** facilita la implementación correcta

### 🔍 Ventajas de MAML:

- ✅ Model-agnostic (funciona con cualquier arquitectura)
- ✅ Adaptación extremadamente rápida (1-5 pasos)
- ✅ No requiere estructuras especiales
- ✅ State-of-the-art en many Few-Shot tasks

### ⚖️ Limitaciones:

- ⚠️ Computacionalmente costoso (gradientes de 2do orden)
- ⚠️ Puede ser inestable sin careful tuning
- ⚠️ Requiere suficiente diversidad en las tareas de entrenamiento

### 🚀 Próximo Tutorial:

En el **Tutorial 05** veremos **Meta-Learning con Memoria**, donde usamos RNNs para que el modelo "recuerde" cómo adaptar sus parámetros.

---

## 🎉 ¡Felicidades!

Has implementado MAML, uno de los algoritmos más importantes e influyentes en Meta-Learning!


<a name='8-1'></a>### 8.1 - Effect of Number of Inner StepsHow many gradient steps should we take during adaptation?

In [ ]:
print("🔬 Testing different numbers of inner steps...\n")inner_steps_options = [1, 3, 5, 10]results_inner_steps = {}for K in inner_steps_options:    print(f"Testing K={K} inner steps...")        # Create and train MAML    model_test = create_maml_model()        # Train for fewer episodes (just for comparison)    for episode in range(100):        task = create_classification_task(n_way=5, k_shot=5, q_query=15)                # Inner loop with K steps        adapted_params = model_test.clone_parameters()        for step in range(K):            logits = model_test.forward_with_params(                task['x_support'],                adapted_params            )            loss = F.cross_entropy(logits, task['y_support'])            grads = torch.autograd.grad(loss, adapted_params, create_graph=True)            adapted_params = [p - 0.01 * g for p, g in zip(adapted_params, grads)]                # Outer loop        query_logits = model_test.forward_with_params(            task['x_query'],            adapted_params        )        meta_loss = F.cross_entropy(query_logits, task['y_query'])                # Update meta-parameters        optimizer = optim.Adam(model_test.parameters(), lr=0.001)        optimizer.zero_grad()        meta_loss.backward()        optimizer.step()        # Evaluate    test_accs = []    for _ in range(50):        task = create_classification_task(n_way=5, k_shot=5, q_query=15)        # ... adapt and evaluate ...        test_accs.append(acc)        results_inner_steps[K] = {        'mean': np.mean(test_accs),        'std': np.std(test_accs)    }        print(f"  Accuracy: {results_inner_steps[K]['mean']:.2%} ± {results_inner_steps[K]['std']:.2%}\n")# Plot resultsfig, ax = plt.subplots(figsize=(10, 6))K_values = list(results_inner_steps.keys())means = [results_inner_steps[k]['mean'] for k in K_values]stds = [results_inner_steps[k]['std'] for k in K_values]ax.errorbar(K_values, means, yerr=stds, marker='o', markersize=8,           linewidth=2, capsize=5, capthick=2)ax.set_xlabel('Number of Inner Steps (K)', fontsize=12)ax.set_ylabel('Test Accuracy', fontsize=12)ax.set_title('Effect of Inner Steps on MAML Performance', fontsize=14)ax.grid(True, alpha=0.3)ax.set_xticks(K_values)plt.tight_layout()plt.show()print("\n📊 Observations:")print("  - More steps generally help (up to a point)")print("  - Diminishing returns after ~5 steps")print("  - Trade-off: more steps = slower training")

<a name='10'></a>## 10 - MAML vs Prototypical Networks: Head-to-HeadLet's compare these two fundamental meta-learning approaches!<table><tr>    <td><b>Aspect</b></td>    <td><b>MAML</b></td>    <td><b>Prototypical Networks</b></td></tr><tr>    <td>**Paradigm**</td>    <td>Optimization-based</td>    <td>Metric-based</td></tr><tr>    <td>**Core Idea**</td>    <td>Learn good initialization</td>    <td>Learn good embedding space</td></tr><tr>    <td>**Adaptation**</td>    <td>Gradient descent steps</td>    <td>Compute prototypes</td></tr><tr>    <td>**Training Speed**</td>    <td>⚠️ Slow (2nd order grads)</td>    <td>✅ Fast</td></tr><tr>    <td>**Inference Speed**</td>    <td>⚠️ Slow (need gradient steps)</td>    <td>✅ Fast (just compute prototypes)</td></tr><tr>    <td>**Memory Usage**</td>    <td>⚠️ High (store computation graph)</td>    <td>✅ Low</td></tr><tr>    <td>**Flexibility**</td>    <td>✅ Works for any differentiable model</td>    <td>⚠️ Requires embedding architecture</td></tr><tr>    <td>**Performance**</td>    <td>✅ Often better (especially few-shot)</td>    <td>✅ Competitive, simpler</td></tr><tr>    <td>**Interpretability**</td>    <td>⚠️ Medium (optimization trajectory)</td>    <td>✅ High (visual embeddings)</td></tr></table>### Practical Recommendation:**Use Prototypical Networks if:**- You need fast training and inference- Limited computational resources- Want interpretable results- Working with image classification primarily**Use MAML if:**- You have computational resources- Need state-of-the-art performance- Working with diverse task types (not just classification)- Can tolerate longer training times

<a name='11'></a>## 11 - Summary and Key Takeaways<font color='blue'>**What you should remember:**- ✅ MAML learns an **initialization** that adapts quickly to new tasks- ✅ Uses **bi-level optimization**: inner loop (adapt) + outer loop (meta-learn)- ✅ Requires **second-order gradients** for full version (or first-order approximation)- ✅ More computationally expensive than metric-based methods, but often better performance- ✅ **Model-agnostic**: works with any differentiable model architecture- ✅ Inner loop hyperparameters (α, K) are crucial for performance</font>### 🎯 Performance SummaryTypical results on few-shot tasks:<table><tr>    <td><b>Dataset</b></td>    <td><b>Scenario</b></td>    <td><b>MAML Accuracy</b></td>    <td><b>vs Random</b></td></tr><tr>    <td>Omniglot</td>    <td>5-way 1-shot</td>    <td>~98.7%</td>    <td>+78.7% over 20%</td></tr><tr>    <td>Omniglot</td>    <td>5-way 5-shot</td>    <td>~99.9%</td>    <td>+79.9% over 20%</td></tr><tr>    <td>Mini-ImageNet</td>    <td>5-way 1-shot</td>    <td>~48.7%</td>    <td>+28.7% over 20%</td></tr><tr>    <td>Mini-ImageNet</td>    <td>5-way 5-shot</td>    <td>~63.1%</td>    <td>+43.1% over 20%</td></tr></table>### 💡 Practical Tips:1. **Start with FOMAML** (first-order) - almost as good, much faster2. **Tune inner LR carefully** - typically 0.01-0.1 works well3. **Use 3-5 inner steps** - more doesn't always help4. **Batch inner updates** - process multiple tasks in parallel5. **Monitor both inner and outer losses** - debugging bi-level optimization### 🚀 Extensions and Variants:- **FOMAML**: First-order approximation (faster)- **Reptile**: Even simpler, no computation graph needed- **MAML++**: Various improvements (MSL, CAVIA, etc.)- **Meta-SGD**: Learn inner learning rate per-parameter- **ANIL**: Almost No Inner Loop - most parameters don't need adaptation### 📚 References:- Original Paper: [Model-Agnostic Meta-Learning (MAML)](https://arxiv.org/abs/1703.03400) (Finn et al., 2017)- FOMAML: Same paper, Appendix A- Reptile: [On First-Order Meta-Learning Algorithms](https://arxiv.org/abs/1803.02999) (Nichol et al., 2018)- Implementation: Uses `higher` library for efficient second-order gradients---## 🎉 Congratulations!You've mastered MAML, one of the most influential meta-learning algorithms! You now understand:- The principle of learning to learn through gradient descent- How bi-level optimization works- The trade-offs between second-order and first-order variants- When to use MAML vs other meta-learning approaches**This is a major milestone in your meta-learning journey!** 🚀